# Model B Training — The Full Brain-Inspired Architecture Family

This notebook trains and compares **every architecture variant brainstormed
in the brain-inspired research thread**, not just Model B on its own. It
replaces the earlier version of this notebook, which:

1. only implemented Model B (base), Fix A, and Fix B,
2. crashed on Fix A/Fix B due to a stale-kernel bug (a class defined in one
   cell wasn't in scope when a later cell ran it, because cells had been
   re-run out of order across kernel restarts — a classic Jupyter trap), and
3. never actually got a result for Fix A or Fix B as a consequence.

**Everything in this notebook is written to run top-to-bottom in one clean
kernel session, in order, with no hidden dependency on cells having been run
out of order before.** Re-run the whole notebook from the top if you ever
touch an earlier cell.

## The 9 configurations trained here

| # | Name | What changed vs. the previous one | Status before this rewrite |
|---|------|-----------------------------------|------------------------------|
| 0 | `baseline` | Plain conv stack, no prediction/gating — the control arm | Trained, 46.72% test acc |
| 1 | `model_b` | Surprise-only propagation (local prediction error, gated) | Trained, 51.48% test acc |
| 2 | `fix_a` | Model B minus the reconstruction loss term | **Crashed, never trained** |
| 3 | `fix_b` | Model B with a deeper 2-layer predictor + higher sparsity weight | **Crashed, never trained** |
| 4 | `model_1` | Model B's local error + one global "dopamine" scalar modulating every gate | **Never coded** |
| 5 | `model_2a` | Model 1 + "serotonin" (environmental-stability estimate) | **Never coded** |
| 6 | `model_2b` | Model 2a + "acetylcholine" + "norepinephrine" (hypothesized sweet spot; specifically fixes the Dead Layer Problem) | **Never coded** |
| 7 | `model_2c` | Model 2b + "cortisol" + "endorphins" (hypothesized to add noise, not help) | **Never coded** |
| 8 | `model_a` | Full biological mapping: thalamic gating, k-WTA sparsity, hippocampal memory, persistent context/working memory | **Never coded** |

Every model below is trained and evaluated through the *exact same*
`train_model()` function (see the "Training & Evaluation" section) — no
per-model copy-pasted training loops, which is what let the original
notebook drift out of sync with itself in the first place.

**Honest caveat, stated once here instead of scattered through every model's
docstring:** the biological neuromodulator systems (dopamine, serotonin,
acetylcholine, norepinephrine, cortisol, endorphins) are literal per-synapse
signalling mechanisms in the brain, updated outside of backpropagation. We
are training everything here with ordinary backprop through small learned
"modulator heads" instead, because that's what a standard PyTorch training
loop can actually optimize end-to-end. Each model's docstring explains the
specific translation being made — these are reasonable, defensible
approximations of the *mechanism*, not literal reproductions of the
biology. Treat Models 1 through 2c as a genuine empirical test of "does
richer global modulation help a surprise-gated network," not as a claim
about how neurons actually work.

## 1. Setup & Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import pandas as pd
import numpy as np
from PIL import Image
import os
import json
import matplotlib.pyplot as plt
from datetime import datetime

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cpu


## 2. Data Loading

Unchanged from the original harness — this part was never broken. Reuses
`data/processed/{train,val,test}.csv`, the same split SkinSense's production
model was trained on, so every variant here is compared on the identical
data SkinSense itself uses.

In [2]:
class SkinDataset(Dataset):
    """Loads images from a CSV with `image_path` and `numeric_label` columns.
    Falls back to a blank image on a read failure instead of crashing the
    whole training run over one bad file (see the SkinSense notebook's
    recurring missing-SCIN-image error — this dataset class is deliberately
    defensive against exactly that)."""
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row['image_path']
        label = row['numeric_label']
        try:
            image = Image.open(image_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (224, 224))
        if self.transform:
            image = self.transform(image)
        return image, label


def get_dataloaders(data_dir, batch_size=32, num_workers=4):
    """Train loader gets augmentation (flips, rotation, color jitter,
    translation); val/test loaders only get resize + normalize, so
    evaluation numbers reflect the model's real generalization, not
    augmentation artifacts."""
    IMG_SIZE = 224

    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    eval_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    train_dataset = SkinDataset(f'{data_dir}/train.csv', transform=train_transform)
    val_dataset = SkinDataset(f'{data_dir}/val.csv', transform=eval_transform)
    test_dataset = SkinDataset(f'{data_dir}/test.csv', transform=eval_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                               num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader

## 3. The Shared Base Component: `PredictiveLayer`

This single module is the mechanism every brain-inspired variant in this
notebook is built on top of (Model A extends it further with thalamic
gating and k-WTA sparsity, but the predictive-coding core is identical).
Get this right once and every model downstream inherits a correct
foundation — get it wrong once and every model downstream inherits the same
bug, which is exactly what happened with the Fix A/B crash.

In [3]:
class PredictiveLayer(nn.Module):
    """
    The core building block of Model B and everything derived from it.

    Biological grounding (predictive coding, Rao & Ballard 1999): each stage
    of cortex doesn't forward raw sensory data upward. It predicts what the
    incoming signal SHOULD be, and only the mismatch between prediction and
    reality ("surprise" / prediction error) is what actually propagates to
    the next stage. If the prediction is perfect, nothing new propagates —
    the brain doesn't waste computation re-processing information it
    already expected. This is also mechanically close to why a dermatologist
    can glance at skin for two seconds and know something is wrong: they
    aren't processing the whole image, they're scanning for deviations from
    what healthy skin should look like.

    Mechanically, five pieces:
      1. `encoder`       - a normal conv block, turns the input into features.
      2. `predictor`     - tries to reconstruct the ORIGINAL input from those
                            features. If the encoder captured the input well,
                            this reconstruction will be close to `x`.
      3. `error`         - x - prediction. This IS the surprise signal.
      4. `error_encoder` - re-encodes the error into the same shape as
                            `encoded`, so it can be gated and passed onward.
      5. `surprise_gate` - a learned sigmoid gate, one value per channel,
                            deciding how much of the error is worth passing
                            on. Near 0 = "I predicted this well, the next
                            layer doesn't need to know." Near 1 = "I was
                            surprised, the next layer needs this."
    """
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self.predictor = nn.Sequential(
            nn.Conv2d(out_channels, in_channels, 1, bias=False),
            nn.BatchNorm2d(in_channels),
        )
        self.error_encoder = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self.surprise_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(out_channels, out_channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        encoded = self.encoder(x)
        prediction = self.predictor(encoded)
        if prediction.shape != x.shape:
            prediction = F.interpolate(prediction, size=x.shape[2:], mode='bilinear', align_corners=False)

        error = x - prediction
        error_encoded = self.error_encoder(error)

        gate = self.surprise_gate(error_encoded)          # (B, C) — one gate value per channel
        gate_map = gate.unsqueeze(-1).unsqueeze(-1)        # (B, C, 1, 1) — broadcastable over space
        gated_error = error_encoded * gate_map

        return gated_error, prediction, error, gate


print("PredictiveLayer defined — shared by Model B, Fix A, Model 1, Model 2a/2b/2c, and Model A")

PredictiveLayer defined — shared by Model B, Fix A, Model 1, Model 2a/2b/2c, and Model A


## 4. The Control Arm: `Baseline`

Every brain-inspired idea below is only interesting if it beats this. A
plain 4-layer conv stack, same depth and comparable capacity, no
prediction, no gating, no sparsity — just to make sure any improvement we
see is actually coming from the brain-inspired mechanism, not just from
"having 4 conv layers."

In [4]:
class StandardConvLayer(nn.Module):
    """Plain conv block: conv -> batchnorm -> relu. No prediction, no gate."""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class Baseline(nn.Module):
    """The control arm. If Model B (or any variant below) doesn't beat this,
    surprise-only propagation isn't buying anything over an ordinary CNN of
    the same depth."""
    def __init__(self, num_classes=7):
        super().__init__()
        self.layer1 = StandardConvLayer(3, 32, stride=1)
        self.layer2 = StandardConvLayer(32, 64, stride=2)
        self.layer3 = StandardConvLayer(64, 128, stride=2)
        self.layer4 = StandardConvLayer(128, 256, stride=2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        pooled = self.global_pool(x).view(x.size(0), -1)
        logits = self.fc(pooled)
        return logits, {"pooled": pooled}


print("Baseline defined")

Baseline defined


## 5. Model B (base) — Surprise-Only Propagation

The original hypothesis: stack four `PredictiveLayer`s. Only the gated
surprise signal moves from one layer to the next — never the raw
activations. This is the variant that was actually trained before (50
epochs, 51.48% test accuracy, +4.76pp over Baseline) — reproduced here
unchanged, just properly commented and guaranteed to sit in dependency
order relative to `PredictiveLayer` above it.

In [5]:
class ModelB(nn.Module):
    """Model B (base): four PredictiveLayers stacked, nothing else."""
    def __init__(self, num_classes=7):
        super().__init__()
        self.layer1 = PredictiveLayer(3, 32, stride=1)
        self.layer2 = PredictiveLayer(32, 64, stride=2)
        self.layer3 = PredictiveLayer(64, 128, stride=2)
        self.layer4 = PredictiveLayer(128, 256, stride=2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        error1, pred1, raw_err1, gate1 = self.layer1(x)
        error2, pred2, raw_err2, gate2 = self.layer2(error1)
        error3, pred3, raw_err3, gate3 = self.layer3(error2)
        error4, pred4, raw_err4, gate4 = self.layer4(error3)

        pooled = self.global_pool(error4).view(x.size(0), -1)
        logits = self.fc(pooled)

        aux = {
            "predictions": [pred1, pred2, pred3, pred4],
            "errors": [raw_err1, raw_err2, raw_err3, raw_err4],
            "gates": [gate1, gate2, gate3, gate4],
            "inputs": [x, error1, error2, error3],   # what each layer actually received
            "pooled": pooled,
        }
        return logits, aux


print("ModelB defined")

ModelB defined


## 6. Model B — Fix A: Remove Reconstruction Loss

**This is the variant that crashed before.** The architecture is identical
to Model B — the fix is entirely in the *loss function* (see Section 13),
which drops the reconstruction term and trains on classification + sparsity
only. The hypothesis being tested: maybe the reconstruction loss was
fighting the classification objective (spending capacity on "predict the
input pixel-for-pixel" instead of "predict the input well enough to know
what's NOT skin-relevant") rather than helping it.

Kept as its own class (rather than just reusing `ModelB`) so its
checkpoints and training history are tracked independently in the
comparison section — this was always the intent, the crash was purely a
notebook cell-ordering bug, not a design flaw.

In [6]:
class ModelBFixA(nn.Module):
    """Fix A: identical architecture to Model B. Paired with `fix_a_loss`
    (Section 13), which is the actual fix — no reconstruction term."""
    def __init__(self, num_classes=7):
        super().__init__()
        self.layer1 = PredictiveLayer(3, 32, stride=1)
        self.layer2 = PredictiveLayer(32, 64, stride=2)
        self.layer3 = PredictiveLayer(64, 128, stride=2)
        self.layer4 = PredictiveLayer(128, 256, stride=2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        error1, pred1, raw_err1, gate1 = self.layer1(x)
        error2, pred2, raw_err2, gate2 = self.layer2(error1)
        error3, pred3, raw_err3, gate3 = self.layer3(error2)
        error4, pred4, raw_err4, gate4 = self.layer4(error3)

        pooled = self.global_pool(error4).view(x.size(0), -1)
        logits = self.fc(pooled)

        aux = {
            "predictions": [pred1, pred2, pred3, pred4],
            "errors": [raw_err1, raw_err2, raw_err3, raw_err4],
            "gates": [gate1, gate2, gate3, gate4],
            "inputs": [x, error1, error2, error3],
            "pooled": pooled,
        }
        return logits, aux


print("ModelBFixA defined — will not crash: PredictiveLayer is already in scope above")

ModelBFixA defined — will not crash: PredictiveLayer is already in scope above


## 7. Model B — Fix B: Better Predictor + Higher Sparsity Weight

**The other variant that crashed before.** Different diagnosis than Fix A:
the base `PredictiveLayer`'s `predictor` is a single 1x1 conv — barely more
than a linear projection. That may not have enough capacity to produce a
*good* prediction, so the "error" it computes is dominated by the
predictor's own weakness rather than genuine surprise in the data. A weak
predictor's error looks statistically similar everywhere, so the gate never
learns to open wide for genuinely surprising inputs and stay shut for
predictable ones — which is exactly the "gates stayed flat around 0.42"
result Model B actually produced.

Fix: give the predictor two conv layers instead of one, and pair it with a
5x higher sparsity-loss weight (`fix_b_loss`, Section 13) to push harder
toward closed gates now that there's a more trustworthy error signal to
sparsify against.

In [7]:
class PredictiveLayerBetterPredictor(nn.Module):
    """Same as PredictiveLayer, except `predictor` gets a second conv layer
    (with its own BatchNorm + ReLU) before projecting back down to the
    input's channel count."""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self.predictor = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, in_channels, 1, bias=False),
            nn.BatchNorm2d(in_channels),
        )
        self.error_encoder = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self.surprise_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(out_channels, out_channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        encoded = self.encoder(x)
        prediction = self.predictor(encoded)
        if prediction.shape != x.shape:
            prediction = F.interpolate(prediction, size=x.shape[2:], mode='bilinear', align_corners=False)
        error = x - prediction
        error_encoded = self.error_encoder(error)
        gate = self.surprise_gate(error_encoded)
        gate_map = gate.unsqueeze(-1).unsqueeze(-1)
        gated_error = error_encoded * gate_map
        return gated_error, prediction, error, gate


class ModelBFixB(nn.Module):
    """Fix B: the same 4-layer stack, using PredictiveLayerBetterPredictor
    instead of the base PredictiveLayer at every layer."""
    def __init__(self, num_classes=7):
        super().__init__()
        self.layer1 = PredictiveLayerBetterPredictor(3, 32, stride=1)
        self.layer2 = PredictiveLayerBetterPredictor(32, 64, stride=2)
        self.layer3 = PredictiveLayerBetterPredictor(64, 128, stride=2)
        self.layer4 = PredictiveLayerBetterPredictor(128, 256, stride=2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        error1, pred1, raw_err1, gate1 = self.layer1(x)
        error2, pred2, raw_err2, gate2 = self.layer2(error1)
        error3, pred3, raw_err3, gate3 = self.layer3(error2)
        error4, pred4, raw_err4, gate4 = self.layer4(error3)

        pooled = self.global_pool(error4).view(x.size(0), -1)
        logits = self.fc(pooled)

        aux = {
            "predictions": [pred1, pred2, pred3, pred4],
            "errors": [raw_err1, raw_err2, raw_err3, raw_err4],
            "gates": [gate1, gate2, gate3, gate4],
            "inputs": [x, error1, error2, error3],
            "pooled": pooled,
        }
        return logits, aux


print("PredictiveLayerBetterPredictor and ModelBFixB defined")

PredictiveLayerBetterPredictor and ModelBFixB defined


## 8. Model 1 — Local Prediction Errors + Global Dopamine

The first genuinely new model in this notebook — this was designed in the
neuroscience chat as "Hypothesis 8" but never actually coded until now.

Architecturally the *same* 4-layer PredictiveLayer stack as Model B — the
local-error mechanism (Hypothesis 7) was never in question. What's new is a
single global scalar, "dopamine", computed once per forward pass from the
network's own pooled representation, answering "how wrong/uncertain does
the network feel about this input, overall?" — and then used to scale every
layer's gate.

Biological grounding: dopamine neurons in the midbrain (VTA / substantia
nigra) fire in proportion to reward-prediction error and broadcast that
signal brain-wide. A LOCAL surprise signal (this layer predicted badly)
should matter more when the SYSTEM AS A WHOLE is also uncertain, and matter
less when the system is otherwise confident — dopamine doesn't replace
local error, it modulates how much local error is allowed to influence what
propagates.

In [8]:
class Model1(nn.Module):
    """
    Model 1: local prediction errors (unchanged from Model B) modulated by
    one global "dopamine" scalar.

    Honest caveat: the real biological rule is a literal per-synapse Hebbian
    update, `delta_w = local_error * dopamine * pre_activity * post_activity`,
    applied outside of backprop. Here we approximate the SPIRIT of that rule
    (local error, globally modulated) with ordinary backprop through a small
    differentiable "dopamine head" — not a literal implementation of it.
    """
    def __init__(self, num_classes=7):
        super().__init__()
        self.layer1 = PredictiveLayer(3, 32, stride=1)
        self.layer2 = PredictiveLayer(32, 64, stride=2)
        self.layer3 = PredictiveLayer(64, 128, stride=2)
        self.layer4 = PredictiveLayer(128, 256, stride=2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)

        # Reads the final pooled representation, outputs one scalar per
        # image in [0, 1]. Small by design: a cheap global read-out, not a
        # second classifier.
        self.dopamine_head = nn.Sequential(
            nn.Linear(256, 32), nn.ReLU(inplace=True), nn.Linear(32, 1), nn.Sigmoid()
        )

    def forward(self, x):
        error1, pred1, raw_err1, gate1 = self.layer1(x)
        error2, pred2, raw_err2, gate2 = self.layer2(error1)
        error3, pred3, raw_err3, gate3 = self.layer3(error2)
        error4, pred4, raw_err4, gate4 = self.layer4(error3)

        pooled = self.global_pool(error4).view(x.size(0), -1)
        logits = self.fc(pooled)

        # Detached: "how wrong do we feel" shouldn't fight the classification
        # gradient flowing through `pooled` via a second path.
        dopamine = self.dopamine_head(pooled.detach())  # (B, 1)

        gates = [gate1, gate2, gate3, gate4]
        # A layer with a big local error still gets DOWN-weighted if the
        # network is otherwise confident; a layer with a small local error
        # still gets a nudge if the network overall is very uncertain.
        modulated_gates = [g * dopamine for g in gates]

        aux = {
            "predictions": [pred1, pred2, pred3, pred4],
            "errors": [raw_err1, raw_err2, raw_err3, raw_err4],
            "gates": gates,                        # raw, for gate-collapse diagnostics
            "modulated_gates": modulated_gates,     # what the sparsity loss actually optimizes
            "inputs": [x, error1, error2, error3],
            "dopamine": dopamine,
            "pooled": pooled,
        }
        return logits, aux


print("Model1 defined")

Model1 defined


## 9. Model 2a — + Serotonin (2 neuromodulators)

Biological grounding: serotonin (raphe nuclei) tracks the brain's estimate
of environmental *stability/predictability*, distinct from dopamine's
moment-to-moment error signal. A stable, predictable environment -> higher
serotonin -> the system trusts its own gating decisions more and updates
less aggressively even when locally surprised, because "surprising" inputs
in an otherwise-volatile training run are more likely noise than signal.

Implementation: serotonin is derived from the *running variance* of the
dopamine signal itself (tracked as a buffer, like BatchNorm tracks running
statistics — not a learned parameter, since "how stable has training been
lately" is a property of training dynamics, not of any single image).

In [9]:
class Model2a(nn.Module):
    """Model 1 + Serotonin. Serotonin dampens how much dopamine is allowed
    to swing the gates, based on how stable the dopamine signal itself has
    been across recent batches."""
    def __init__(self, num_classes=7, serotonin_momentum=0.95):
        super().__init__()
        self.layer1 = PredictiveLayer(3, 32, stride=1)
        self.layer2 = PredictiveLayer(32, 64, stride=2)
        self.layer3 = PredictiveLayer(64, 128, stride=2)
        self.layer4 = PredictiveLayer(128, 256, stride=2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)

        self.dopamine_head = nn.Sequential(
            nn.Linear(256, 32), nn.ReLU(inplace=True), nn.Linear(32, 1), nn.Sigmoid()
        )

        # Running dopamine statistics -> derive serotonin from them.
        # Buffers, not parameters: saved/loaded with the model, but the
        # optimizer never touches them directly.
        self.serotonin_momentum = serotonin_momentum
        self.register_buffer("running_dopamine_mean", torch.tensor(0.5))
        self.register_buffer("running_dopamine_var", torch.tensor(0.05))

    def forward(self, x):
        error1, pred1, raw_err1, gate1 = self.layer1(x)
        error2, pred2, raw_err2, gate2 = self.layer2(error1)
        error3, pred3, raw_err3, gate3 = self.layer3(error2)
        error4, pred4, raw_err4, gate4 = self.layer4(error3)

        pooled = self.global_pool(error4).view(x.size(0), -1)
        logits = self.fc(pooled)
        dopamine = self.dopamine_head(pooled.detach())

        if self.training:
            batch_mean = dopamine.mean().detach()
            batch_var = dopamine.var(unbiased=False).detach()
            m = self.serotonin_momentum
            self.running_dopamine_mean.mul_(m).add_(batch_mean * (1 - m))
            self.running_dopamine_var.mul_(m).add_(batch_var * (1 - m))

        # High recent variance -> low serotonin (volatile). Simple decreasing
        # squash into [0, 1].
        serotonin = torch.exp(-5.0 * self.running_dopamine_var).clamp(0.0, 1.0)

        # Blend "raw dopamine modulation" with "trust the local gate as-is
        # (multiplier of 1.0)" in proportion to how stable things have been.
        effective_modulation = dopamine * (1 - serotonin) + 1.0 * serotonin

        gates = [gate1, gate2, gate3, gate4]
        modulated_gates = [g * effective_modulation for g in gates]

        aux = {
            "predictions": [pred1, pred2, pred3, pred4],
            "errors": [raw_err1, raw_err2, raw_err3, raw_err4],
            "gates": gates,
            "modulated_gates": modulated_gates,
            "inputs": [x, error1, error2, error3],
            "dopamine": dopamine,
            "serotonin": serotonin,
            "pooled": pooled,
        }
        return logits, aux


print("Model2a defined")

Model2a defined


## 10. Model 2b — + Acetylcholine + Norepinephrine (4 neuromodulators)

**The variant hypothesized as the sweet spot**, and the direct fix for what
Model 1 could only patch with an ad-hoc constant ("let silent layers take
small non-zero updates regardless of activity"). Here the fix is principled:

- **Acetylcholine**: attention/salience — "how much does this particular
  error matter, independent of its raw size." Computed from *global* pooled
  features, not from any single layer's own (possibly near-zero) gate — this
  is specifically what lets it rescue a layer that's gone quiet. It's added
  as an *additive floor* on top of the dopamine/serotonin multiplier, so a
  silenced layer is never fully unreachable. This is the Dead Layer Problem
  fix the original research thread never got to actually build.
- **Norepinephrine**: novelty/urgency — how far this batch's dopamine
  jumped from the running mean (a spike detector), further raising the
  acetylcholine floor when something unusual is happening.

In [10]:
class Model2b(nn.Module):
    """Model 2a + Acetylcholine + Norepinephrine. This is the variant that
    actually solves the Dead Layer Problem, via a global-context-driven
    additive floor on the gate rather than a fixed constant."""
    def __init__(self, num_classes=7, serotonin_momentum=0.95):
        super().__init__()
        self.layer1 = PredictiveLayer(3, 32, stride=1)
        self.layer2 = PredictiveLayer(32, 64, stride=2)
        self.layer3 = PredictiveLayer(64, 128, stride=2)
        self.layer4 = PredictiveLayer(128, 256, stride=2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)

        self.dopamine_head = nn.Sequential(
            nn.Linear(256, 32), nn.ReLU(inplace=True), nn.Linear(32, 1), nn.Sigmoid()
        )
        # Acetylcholine reads GLOBAL pooled features, not any per-layer gate —
        # this is what lets it rescue a silenced layer.
        self.acetylcholine_head = nn.Sequential(
            nn.Linear(256, 32), nn.ReLU(inplace=True), nn.Linear(32, 1), nn.Sigmoid()
        )

        self.serotonin_momentum = serotonin_momentum
        self.register_buffer("running_dopamine_mean", torch.tensor(0.5))
        self.register_buffer("running_dopamine_var", torch.tensor(0.05))

    def forward(self, x):
        error1, pred1, raw_err1, gate1 = self.layer1(x)
        error2, pred2, raw_err2, gate2 = self.layer2(error1)
        error3, pred3, raw_err3, gate3 = self.layer3(error2)
        error4, pred4, raw_err4, gate4 = self.layer4(error3)

        pooled = self.global_pool(error4).view(x.size(0), -1)
        logits = self.fc(pooled)

        dopamine = self.dopamine_head(pooled.detach())
        acetylcholine = self.acetylcholine_head(pooled.detach())

        if self.training:
            batch_mean = dopamine.mean().detach()
            batch_var = dopamine.var(unbiased=False).detach()
            m = self.serotonin_momentum
            prev_mean = self.running_dopamine_mean.clone()
            self.running_dopamine_mean.mul_(m).add_(batch_mean * (1 - m))
            self.running_dopamine_var.mul_(m).add_(batch_var * (1 - m))
        else:
            prev_mean = self.running_dopamine_mean

        serotonin = torch.exp(-5.0 * self.running_dopamine_var).clamp(0.0, 1.0)
        # Norepinephrine: how far THIS batch's dopamine jumped from the
        # running mean — a spike detector for "something changed."
        norepinephrine = (dopamine.mean().detach() - prev_mean).abs().clamp(0.0, 1.0)

        effective_modulation = dopamine * (1 - serotonin) + 1.0 * serotonin
        # Additive floor (not multiplicative) — capable of rescuing a gate
        # that's collapsed to near-zero, unlike everything above it.
        salience_floor = (acetylcholine * (0.5 + 0.5 * norepinephrine)) * 0.3

        gates = [gate1, gate2, gate3, gate4]
        modulated_gates = [
            torch.clamp(g * effective_modulation + salience_floor, 0.0, 1.0) for g in gates
        ]

        aux = {
            "predictions": [pred1, pred2, pred3, pred4],
            "errors": [raw_err1, raw_err2, raw_err3, raw_err4],
            "gates": gates,
            "modulated_gates": modulated_gates,
            "inputs": [x, error1, error2, error3],
            "dopamine": dopamine, "serotonin": serotonin,
            "acetylcholine": acetylcholine, "norepinephrine": norepinephrine,
            "pooled": pooled,
        }
        return logits, aux


print("Model2b defined")

Model2b defined


## 11. Model 2c — + Cortisol + Endorphins (6 neuromodulators)

Exists specifically to *test* the hypothesis (never confirmed) that 2b is
the sweet spot and additional modulators mostly add noise. Same mechanism
family as 2b so the comparison is fair:

- **Cortisol**: sustained stress, not a single spike — modeled as a slow
  running average of norepinephrine ("has the environment been demanding
  for a while"), which further dampens serotonin's stabilizing effect
  during prolonged high-error periods.
- **Endorphins**: reward intensity, approximated from how confident the
  current prediction is (max softmax probability) — dampens the salience
  floor when the network is already confident, on the reasoning that a
  confident, rewarded state shouldn't be reactivating quiet layers.

If this variant does **not** beat 2b in the comparison run, that is the
*expected* outcome per the original hypothesis, not a bug.

In [11]:
class Model2c(nn.Module):
    """Model 2b + Cortisol + Endorphins. Built to test whether more
    modulators keep helping or start hurting."""
    def __init__(self, num_classes=7, serotonin_momentum=0.95, cortisol_momentum=0.9):
        super().__init__()
        self.layer1 = PredictiveLayer(3, 32, stride=1)
        self.layer2 = PredictiveLayer(32, 64, stride=2)
        self.layer3 = PredictiveLayer(64, 128, stride=2)
        self.layer4 = PredictiveLayer(128, 256, stride=2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)

        self.dopamine_head = nn.Sequential(nn.Linear(256, 32), nn.ReLU(inplace=True), nn.Linear(32, 1), nn.Sigmoid())
        self.acetylcholine_head = nn.Sequential(nn.Linear(256, 32), nn.ReLU(inplace=True), nn.Linear(32, 1), nn.Sigmoid())

        self.serotonin_momentum = serotonin_momentum
        self.cortisol_momentum = cortisol_momentum
        self.register_buffer("running_dopamine_mean", torch.tensor(0.5))
        self.register_buffer("running_dopamine_var", torch.tensor(0.05))
        self.register_buffer("running_norepinephrine", torch.tensor(0.0))

    def forward(self, x):
        error1, pred1, raw_err1, gate1 = self.layer1(x)
        error2, pred2, raw_err2, gate2 = self.layer2(error1)
        error3, pred3, raw_err3, gate3 = self.layer3(error2)
        error4, pred4, raw_err4, gate4 = self.layer4(error3)

        pooled = self.global_pool(error4).view(x.size(0), -1)
        logits = self.fc(pooled)

        dopamine = self.dopamine_head(pooled.detach())
        acetylcholine = self.acetylcholine_head(pooled.detach())
        # Endorphins: proxy for "reward" — confidence in the current
        # top prediction.
        endorphins = F.softmax(logits.detach(), dim=1).max(dim=1, keepdim=True).values

        if self.training:
            batch_mean = dopamine.mean().detach()
            batch_var = dopamine.var(unbiased=False).detach()
            m = self.serotonin_momentum
            prev_mean = self.running_dopamine_mean.clone()
            self.running_dopamine_mean.mul_(m).add_(batch_mean * (1 - m))
            self.running_dopamine_var.mul_(m).add_(batch_var * (1 - m))
        else:
            prev_mean = self.running_dopamine_mean

        serotonin = torch.exp(-5.0 * self.running_dopamine_var).clamp(0.0, 1.0)
        norepinephrine = (dopamine.mean().detach() - prev_mean).abs().clamp(0.0, 1.0)

        if self.training:
            cm = self.cortisol_momentum
            self.running_norepinephrine.mul_(cm).add_(norepinephrine * (1 - cm))
        cortisol = self.running_norepinephrine.clamp(0.0, 1.0)

        # Cortisol suppresses serotonin's calming effect during sustained
        # stress; endorphins suppress the salience floor when things
        # already feel rewarding/confident.
        effective_serotonin = serotonin * (1 - cortisol)
        effective_modulation = dopamine * (1 - effective_serotonin) + 1.0 * effective_serotonin
        salience_floor = (acetylcholine * (0.5 + 0.5 * norepinephrine) * (1 - 0.5 * endorphins)) * 0.3

        gates = [gate1, gate2, gate3, gate4]
        modulated_gates = [
            torch.clamp(g * effective_modulation + salience_floor, 0.0, 1.0) for g in gates
        ]

        aux = {
            "predictions": [pred1, pred2, pred3, pred4],
            "errors": [raw_err1, raw_err2, raw_err3, raw_err4],
            "gates": gates,
            "modulated_gates": modulated_gates,
            "inputs": [x, error1, error2, error3],
            "dopamine": dopamine, "serotonin": serotonin, "acetylcholine": acetylcholine,
            "norepinephrine": norepinephrine, "cortisol": cortisol, "endorphins": endorphins,
            "pooled": pooled,
        }
        return logits, aux


print("Model2c defined")

Model2c defined


## 12. Model A — Full Brain Architecture

The biologically-faithful branch of this research (as opposed to Model B's
deliberately stripped-down "surprise only" hypothesis). This is the most
complex model in the notebook, and honestly the least validated — every
piece below is a reasonable but non-unique translation of a biological
mechanism into a differentiable module. Treat its results as "does
combining everything help, hurt, or wash out," not as a definitive
brain-architecture result.

Four sub-components, built first, then composed into `ModelA`:

- **`KWinnersTakeAll`** — LOCAL, per-layer sparsity (top-k% of channels per
  spatial location survive, the rest are zeroed). Deliberately *not* a
  global top-k sort — that distinction mattered from the very first message
  of this whole thread ("does the brain actually do that though?" — no,
  real lateral inhibition in cortex is local, from neighboring inhibitory
  interneurons).
- **`ThalamicGate`** — filters the input to each layer based on a running
  *context* vector (see below), the way the thalamus gates what reaches
  cortex based on attention/current goals, not just raw sensory data.
- **`HippocampalMemory`** — a small external memory bank queried at
  inference, the analog of hippocampal fast memory/pattern completion, as
  distinct from the slow memory baked into conv weights over many epochs.
- **Persistent context (prefrontal working memory + top-down feedback)** —
  a vector updated via a `GRUCell` after every layer, carrying a running
  summary of everything processed so far in this pass forward through the
  layers below it. This doubles as this notebook's approximation of
  top-down feedback: a literal second top-down pass (deeper layers
  correcting earlier ones after a full first pass) would need real
  recurrence across the whole network, which is left out here for
  tractability — the context vector is the single-pass approximation.

In [12]:
class KWinnersTakeAll(nn.Module):
    """LOCAL sparsity: keeps only the top-k% of channel activations at each
    spatial location, zeroes the rest. Approximates a small neighborhood of
    cortical neurons suppressing weaker neighbors after one fires strongly —
    deliberately not a global top-k across the whole feature map."""
    def __init__(self, k_percent=0.2):
        super().__init__()
        self.k_percent = k_percent

    def forward(self, x):
        b, c, h, w = x.shape
        k = max(1, int(c * self.k_percent))
        topk_vals, topk_idx = x.topk(k, dim=1)
        mask = torch.zeros_like(x).scatter_(1, topk_idx, 1.0)
        return x * mask


class ThalamicGate(nn.Module):
    """Context-conditioned input filter. `context` is the running summary
    vector (see ModelA.forward) built up from everything the network has
    processed so far in this pass — used to modulate this layer's input
    channel-wise before it reaches the PredictiveLayer."""
    def __init__(self, in_channels, context_dim):
        super().__init__()
        self.context_to_gate = nn.Sequential(nn.Linear(context_dim, in_channels), nn.Sigmoid())

    def forward(self, x, context):
        gate = self.context_to_gate(context)
        gate = gate.unsqueeze(-1).unsqueeze(-1)
        return x * gate


class HippocampalMemory(nn.Module):
    """A fixed-size bank of learned memory slots. A query (the image's
    pooled features) retrieves a weighted blend of the slots it's most
    similar to via attention; the retrieved memory is fed into the
    classifier alongside the image's own features."""
    def __init__(self, feature_dim, num_slots=64):
        super().__init__()
        self.memory = nn.Parameter(torch.randn(num_slots, feature_dim) * 0.02)
        self.query_proj = nn.Linear(feature_dim, feature_dim)

    def forward(self, query):
        q = self.query_proj(query)                                          # (B, D)
        attn = F.softmax(q @ self.memory.T / (q.shape[-1] ** 0.5), dim=-1)    # (B, num_slots)
        retrieved = attn @ self.memory                                        # (B, D)
        return retrieved, attn


class ModelA(nn.Module):
    """
    Full Brain Architecture: thalamic gating -> predictive coding -> local
    k-WTA sparsity, at every layer, with a persistent GRU-updated context
    vector carrying working-memory/top-down influence forward through
    depth, and a hippocampal memory bank queried once at the end.
    """
    def __init__(self, num_classes=7, context_dim=64, memory_slots=64, kwta_percent=0.2):
        super().__init__()
        channels = [3, 32, 64, 128, 256]
        strides = [1, 2, 2, 2]

        self.thalamic_gates = nn.ModuleList([ThalamicGate(channels[i], context_dim) for i in range(4)])
        self.predictive_layers = nn.ModuleList(
            [PredictiveLayer(channels[i], channels[i + 1], stride=strides[i]) for i in range(4)]
        )
        self.kwta_layers = nn.ModuleList([KWinnersTakeAll(kwta_percent) for _ in range(4)])

        # Persistent context: both "prefrontal working memory" and the
        # channel carrying a top-down-ish influence forward through depth.
        self.context_dim = context_dim
        self.context_update = nn.ModuleList([nn.GRUCell(channels[i + 1], context_dim) for i in range(4)])

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.hippocampus = HippocampalMemory(256, num_slots=memory_slots)
        self.dopamine_head = nn.Sequential(
            nn.Linear(256, 32), nn.ReLU(inplace=True), nn.Linear(32, 1), nn.Sigmoid()
        )

        # Classifier reads BOTH the image's own pooled features and the
        # hippocampal-retrieved memory, concatenated.
        self.fc = nn.Linear(256 + 256, num_classes)

    def forward(self, x):
        b = x.size(0)
        context = torch.zeros(b, self.context_dim, device=x.device, dtype=x.dtype)

        current = x
        predictions, raw_errors, gates, layer_inputs = [], [], [], []
        for i in range(4):
            gated_input = self.thalamic_gates[i](current, context)
            layer_inputs.append(gated_input)                       # what this layer actually predicts against

            error, prediction, raw_error, gate = self.predictive_layers[i](gated_input)
            sparse_error = self.kwta_layers[i](error)

            # Update the persistent context from a pooled summary of this
            # layer's output — this is what makes deeper layers' thalamic
            # gates aware of everything processed so far.
            pooled_here = F.adaptive_avg_pool2d(sparse_error, 1).view(b, -1)
            context = self.context_update[i](pooled_here, context)

            predictions.append(prediction)
            raw_errors.append(raw_error)
            gates.append(gate)
            current = sparse_error

        pooled = self.global_pool(current).view(b, -1)
        retrieved_memory, memory_attn = self.hippocampus(pooled)
        combined = torch.cat([pooled, retrieved_memory], dim=1)
        logits = self.fc(combined)

        dopamine = self.dopamine_head(pooled.detach())
        modulated_gates = [g * dopamine for g in gates]

        aux = {
            "predictions": predictions,
            "errors": raw_errors,
            "gates": gates,
            "modulated_gates": modulated_gates,
            "inputs": layer_inputs,               # aligned 1:1 with `predictions`, post-thalamic-gating
            "dopamine": dopamine,
            "memory_attention": memory_attn,
            "context": context,
            "pooled": pooled,
        }
        return logits, aux


print("KWinnersTakeAll, ThalamicGate, HippocampalMemory, and ModelA defined")

KWinnersTakeAll, ThalamicGate, HippocampalMemory, and ModelA defined


## 13. Loss Functions

One shared loss for every PredictiveLayer-based model, plus the two
Fix-A/Fix-B variants that deliberately change which terms are included.
Consolidated here (rather than copy-pasted per model, which is part of how
the original notebook drifted) so every model's loss is one of exactly
three well-understood functions.

In [13]:
def baseline_loss(logits, targets, aux):
    """Plain cross-entropy — the control arm gets no auxiliary terms."""
    ce = F.cross_entropy(logits, targets)
    return {"total": ce, "classification": ce.item()}


def predictive_coding_loss(logits, targets, aux, w_class=1.0, w_recon=0.1, w_sparse=0.01):
    """
    Shared loss for every PredictiveLayer-based model (Model B, Model 1,
    Models 2a-2c, Model A). Three terms:

      1. classification  - the actual task.
      2. reconstruction  - how well each layer's `predictor` reconstructed
                            its own input. Directly trains the predictive-
                            coding mechanism (a layer that can't predict its
                            input can't produce a meaningful surprise signal).
      3. sparsity        - mean gate activation, pushed DOWN. If the
                            mechanism works, this should fall as training
                            progresses (predictable input -> closed gates ->
                            less computation forwarded). Measured on
                            `modulated_gates` where a model provides them
                            (Model 1 onward), so the neuromodulator signal is
                            actually part of what gets optimized, not just
                            logged for inspection.
    """
    class_loss = F.cross_entropy(logits, targets)

    recon_loss = 0.0
    for pred, target_input in zip(aux["predictions"], aux["inputs"]):
        if pred.shape != target_input.shape:
            pred = F.interpolate(pred, size=target_input.shape[2:], mode='bilinear', align_corners=False)
        recon_loss = recon_loss + F.mse_loss(pred, target_input)
    recon_loss = recon_loss / len(aux["predictions"])

    gate_source = aux.get("modulated_gates", aux["gates"])
    sparse_loss = sum(g.mean() for g in gate_source) / len(gate_source)

    total = w_class * class_loss + w_recon * recon_loss + w_sparse * sparse_loss
    return {
        "total": total,
        "classification": class_loss.item(),
        "reconstruction": recon_loss.item(),
        "sparsity": sparse_loss.item(),
    }


def fix_a_loss(logits, targets, aux, w_class=1.0, w_sparse=0.01):
    """Fix A: classification + sparsity only — no reconstruction term."""
    class_loss = F.cross_entropy(logits, targets)
    gate_source = aux.get("modulated_gates", aux["gates"])
    sparse_loss = sum(g.mean() for g in gate_source) / len(gate_source)
    total = w_class * class_loss + w_sparse * sparse_loss
    return {"total": total, "classification": class_loss.item(), "sparsity": sparse_loss.item()}


def fix_b_loss(logits, targets, aux, w_class=1.0, w_recon=0.1, w_sparse=0.05):
    """Fix B: same three terms as the base predictive-coding loss, but with
    a 5x higher sparsity weight (0.05 vs 0.01) — see Section 7 for why."""
    return predictive_coding_loss(logits, targets, aux, w_class, w_recon, w_sparse)


print("Loss functions defined: baseline_loss, predictive_coding_loss, fix_a_loss, fix_b_loss")

Loss functions defined: baseline_loss, predictive_coding_loss, fix_a_loss, fix_b_loss


## 14. Training & Evaluation Harness

One `train_epoch`, one `eval_epoch`, one `train_model` — used for **all 9**
configurations. This is the actual structural fix for what broke Fix A/B:
the original notebook trained each model with its own hand-copied block of
cells, which is exactly how one model's class definition ended up out of
scope for another's training cell after a kernel restart. A single shared
function can't have that bug — every model goes through the identical code
path, so if it works once, it works for all nine.

In [14]:
def train_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    gate_activations = []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        logits, aux = model(images)
        loss_dict = loss_fn(logits, labels, aux)
        loss = loss_dict["total"]

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        _, predicted = logits.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
        if "gates" in aux:
            gate_activations.extend([g.mean().item() for g in aux["gates"]])

    avg_gate = float(np.mean(gate_activations)) if gate_activations else 0.0
    return total_loss / len(loader), correct / total, avg_gate


def eval_epoch(model, loader, loss_fn):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    gate_activations = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits, aux = model(images)
            loss_dict = loss_fn(logits, labels, aux)
            loss_val = loss_dict["total"]
            total_loss += loss_val.item() if isinstance(loss_val, torch.Tensor) else loss_val
            _, predicted = logits.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
            if "gates" in aux:
                gate_activations.extend([g.mean().item() for g in aux["gates"]])

    avg_gate = float(np.mean(gate_activations)) if gate_activations else 0.0
    return total_loss / len(loader), correct / total, avg_gate


def train_model(name, model_cls, loss_fn, train_loader, val_loader, test_loader,
                 epochs, output_root, lr=1e-3, model_kwargs=None):
    """Builds, trains, and checkpoints ONE model by name. Every one of the 9
    training-run cells below is a one-line call into this function — that's
    the whole point: no per-model copy-pasted loop to drift out of sync."""
    model_kwargs = model_kwargs or {}
    model = model_cls(num_classes=7, **model_kwargs).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    output_dir = f"{output_root}/{name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(output_dir, exist_ok=True)

    history = {
        "train_loss": [], "train_acc": [], "train_gate": [],
        "val_loss": [], "val_acc": [], "val_gate": [],
        "test_loss": [], "test_acc": [], "test_gate": [],
    }
    best_val_acc, best_epoch = 0.0, 0

    n_params = sum(p.numel() for p in model.parameters())
    print(f"\n{'='*80}\nTraining {name} ({n_params:,} params) for {epochs} epochs\n{'='*80}\n")

    for epoch in range(epochs):
        train_loss, train_acc, train_gate = train_epoch(model, train_loader, optimizer, loss_fn)
        val_loss, val_acc, val_gate = eval_epoch(model, val_loader, loss_fn)
        test_loss, test_acc, test_gate = eval_epoch(model, test_loader, loss_fn)
        scheduler.step()

        history["train_loss"].append(train_loss); history["train_acc"].append(train_acc); history["train_gate"].append(train_gate)
        history["val_loss"].append(val_loss); history["val_acc"].append(val_acc); history["val_gate"].append(val_gate)
        history["test_loss"].append(test_loss); history["test_acc"].append(test_acc); history["test_gate"].append(test_gate)

        if val_acc > best_val_acc:
            best_val_acc, best_epoch = val_acc, epoch
            torch.save(model.state_dict(), f"{output_dir}/best_model.pth")

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | Train: {train_acc:.4f} acc {train_loss:.4f} loss | "
                  f"Val: {val_acc:.4f} acc {val_loss:.4f} loss | Test: {test_acc:.4f} acc | gate: {val_gate:.4f}")

    print(f"\nBest validation accuracy: {best_val_acc:.4f} at epoch {best_epoch+1}")
    with open(f"{output_dir}/history.json", "w") as f:
        json.dump(history, f, indent=2)

    return model, history, output_dir


print("train_epoch, eval_epoch, and train_model defined")

train_epoch, eval_epoch, and train_model defined


## 15. Configuration

Shared config for every training run, plus the registry mapping each of
the 9 names to its class, loss function, and any extra constructor kwargs.
This is the single place to change epochs/batch size/lr for every model at
once — and the single place to see, at a glance, exactly what's about to
run.

In [15]:
DATA_DIR = '../data/processed'
OUTPUT_DIR = '../models/model_b'
EPOCHS = 50
BATCH_SIZE = 32
LR = 1e-3
NUM_WORKERS = 4

# name -> (class, loss function, extra constructor kwargs)
MODEL_REGISTRY = {
    "baseline": (Baseline,   baseline_loss,          {}),
    "model_b":  (ModelB,     predictive_coding_loss, {}),
    "fix_a":    (ModelBFixA, fix_a_loss,              {}),
    "fix_b":    (ModelBFixB, fix_b_loss,              {}),
    "model_1":  (Model1,     predictive_coding_loss, {}),
    "model_2a": (Model2a,    predictive_coding_loss, {}),
    "model_2b": (Model2b,    predictive_coding_loss, {}),
    "model_2c": (Model2c,    predictive_coding_loss, {}),
    "model_a":  (ModelA,     predictive_coding_loss, {}),
}

print("Configuration:")
print(f"  Data dir: {DATA_DIR}")
print(f"  Output dir: {OUTPUT_DIR}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LR}")
print(f"  Models registered: {list(MODEL_REGISTRY.keys())}")

Configuration:
  Data dir: ../data/processed
  Output dir: ../models/model_b
  Epochs: 50
  Batch size: 32
  Learning rate: 0.001
  Models registered: ['baseline', 'model_b', 'fix_a', 'fix_b', 'model_1', 'model_2a', 'model_2b', 'model_2c', 'model_a']


In [16]:
print(f"Loading data from {DATA_DIR}...")
train_loader, val_loader, test_loader = get_dataloaders(DATA_DIR, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Loading data from ../data/processed...


FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/train.csv'

## 16. Training Runs

In [ ]:
name = "baseline"
model_cls, loss_fn, model_kwargs = MODEL_REGISTRY[name]
model_baseline, history_baseline, dir_baseline = train_model(
    name, model_cls, loss_fn, train_loader, val_loader, test_loader,
    epochs=EPOCHS, output_root=OUTPUT_DIR, lr=LR, model_kwargs=model_kwargs,
)
print(f"\n{name} checkpoints saved to: {dir_baseline}")

In [ ]:
name = "model_b"
model_cls, loss_fn, model_kwargs = MODEL_REGISTRY[name]
model_model_b, history_model_b, dir_model_b = train_model(
    name, model_cls, loss_fn, train_loader, val_loader, test_loader,
    epochs=EPOCHS, output_root=OUTPUT_DIR, lr=LR, model_kwargs=model_kwargs,
)
print(f"\n{name} checkpoints saved to: {dir_model_b}")

In [ ]:
name = "fix_a"
model_cls, loss_fn, model_kwargs = MODEL_REGISTRY[name]
model_fix_a, history_fix_a, dir_fix_a = train_model(
    name, model_cls, loss_fn, train_loader, val_loader, test_loader,
    epochs=EPOCHS, output_root=OUTPUT_DIR, lr=LR, model_kwargs=model_kwargs,
)
print(f"\n{name} checkpoints saved to: {dir_fix_a}")

In [ ]:
name = "fix_b"
model_cls, loss_fn, model_kwargs = MODEL_REGISTRY[name]
model_fix_b, history_fix_b, dir_fix_b = train_model(
    name, model_cls, loss_fn, train_loader, val_loader, test_loader,
    epochs=EPOCHS, output_root=OUTPUT_DIR, lr=LR, model_kwargs=model_kwargs,
)
print(f"\n{name} checkpoints saved to: {dir_fix_b}")

In [ ]:
name = "model_1"
model_cls, loss_fn, model_kwargs = MODEL_REGISTRY[name]
model_model_1, history_model_1, dir_model_1 = train_model(
    name, model_cls, loss_fn, train_loader, val_loader, test_loader,
    epochs=EPOCHS, output_root=OUTPUT_DIR, lr=LR, model_kwargs=model_kwargs,
)
print(f"\n{name} checkpoints saved to: {dir_model_1}")

In [ ]:
name = "model_2a"
model_cls, loss_fn, model_kwargs = MODEL_REGISTRY[name]
model_model_2a, history_model_2a, dir_model_2a = train_model(
    name, model_cls, loss_fn, train_loader, val_loader, test_loader,
    epochs=EPOCHS, output_root=OUTPUT_DIR, lr=LR, model_kwargs=model_kwargs,
)
print(f"\n{name} checkpoints saved to: {dir_model_2a}")

In [ ]:
name = "model_2b"
model_cls, loss_fn, model_kwargs = MODEL_REGISTRY[name]
model_model_2b, history_model_2b, dir_model_2b = train_model(
    name, model_cls, loss_fn, train_loader, val_loader, test_loader,
    epochs=EPOCHS, output_root=OUTPUT_DIR, lr=LR, model_kwargs=model_kwargs,
)
print(f"\n{name} checkpoints saved to: {dir_model_2b}")

In [ ]:
name = "model_2c"
model_cls, loss_fn, model_kwargs = MODEL_REGISTRY[name]
model_model_2c, history_model_2c, dir_model_2c = train_model(
    name, model_cls, loss_fn, train_loader, val_loader, test_loader,
    epochs=EPOCHS, output_root=OUTPUT_DIR, lr=LR, model_kwargs=model_kwargs,
)
print(f"\n{name} checkpoints saved to: {dir_model_2c}")

In [ ]:
name = "model_a"
model_cls, loss_fn, model_kwargs = MODEL_REGISTRY[name]
model_model_a, history_model_a, dir_model_a = train_model(
    name, model_cls, loss_fn, train_loader, val_loader, test_loader,
    epochs=EPOCHS, output_root=OUTPUT_DIR, lr=LR, model_kwargs=model_kwargs,
)
print(f"\n{name} checkpoints saved to: {dir_model_a}")

## 17. Comparison: All 9 Configurations

Run after every training cell above has completed. Builds one table across
all 9 histories instead of the original notebook's hand-written two-model
comparison — add a 10th model to `MODEL_REGISTRY` and this cell covers it
automatically, no edits needed here.

In [ ]:
histories = {
    "baseline": history_baseline, "model_b": history_model_b,
    "fix_a": history_fix_a, "fix_b": history_fix_b,
    "model_1": history_model_1, "model_2a": history_model_2a,
    "model_2b": history_model_2b, "model_2c": history_model_2c,
    "model_a": history_model_a,
}

print(f"{'='*90}\nCOMPARISON: All 9 Configurations\n{'='*90}\n")
print(f"{'Model':<10} {'Best Val':>10} {'Test Acc':>10} {'Avg Gate':>10} {'Params':>12}")
print("-" * 56)

rows = []
for name, h in histories.items():
    best_idx = int(np.argmax(h["val_acc"]))
    avg_gate = float(np.mean(h["val_gate"])) if any(h["val_gate"]) else float("nan")
    rows.append((name, h["val_acc"][best_idx], h["test_acc"][best_idx], avg_gate))
    print(f"{name:<10} {h['val_acc'][best_idx]:>10.4f} {h['test_acc'][best_idx]:>10.4f} {avg_gate:>10.4f}")

baseline_test = histories["baseline"]["test_acc"][int(np.argmax(histories["baseline"]["val_acc"]))]
print(f"\n{'='*90}")
print("Improvement over Baseline (test acc):")
for name, _, test_acc, _ in rows:
    if name == "baseline":
        continue
    print(f"  {name:<10} {test_acc - baseline_test:+.4f}")
print(f"{'='*90}")

In [ ]:
# Gate activation over training, all 9 on one plot — the Dead Layer Problem
# and gate-collapse are both visible directly here: a model that's actually
# learning to sparsify shows gates trending down; a collapsed model shows
# gates pinned near 0 or 1 from the first few epochs onward.
fig, ax = plt.subplots(figsize=(12, 6))
for name, h in histories.items():
    if any(h["val_gate"]):
        ax.plot(h["val_gate"], label=name, linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean gate activation (val)")
ax.set_title("Gate activation across all brain-inspired variants")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/all_models_gate_comparison.png")
plt.show()
print(f"Saved to {OUTPUT_DIR}/all_models_gate_comparison.png")

In [ ]:
# Val accuracy over training, all 9 on one plot.
fig, ax = plt.subplots(figsize=(12, 6))
for name, h in histories.items():
    ax.plot(h["val_acc"], label=name, linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation accuracy")
ax.set_title("Validation accuracy across all 9 configurations")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/all_models_accuracy_comparison.png")
plt.show()
print(f"Saved to {OUTPUT_DIR}/all_models_accuracy_comparison.png")

## 18. Key Insights

**Fill this in after running Section 16.** Framework for reading the
results, not a conclusion — nothing has been trained yet as of this
rewrite.

**Did the neuromodulator escalation (Model 1 -> 2a -> 2b -> 2c) help?**
- If test accuracy rises monotonically through 2b then falls at 2c: the
  original hypothesis (2b is the sweet spot) holds.
- If it keeps rising through 2c: more modulation helps more than expected —
  worth trying a Model 2d.
- If it's flat or noisy the whole way: the neuromodulator framing isn't
  doing real work here, and the gains (if any) are coming from somewhere
  else — check whether `dopamine`/`acetylcholine` head outputs are actually
  varying per-input or have collapsed to a near-constant value (a real
  failure mode for a small sigmoid head with no direct supervision).

**Did Fix A or Fix B actually beat base Model B?**
- Fix A tests "was the reconstruction loss hurting classification."
- Fix B tests "was the predictor too weak to produce a meaningful error
  signal." Compare their `avg_gate` against Model B's ~0.42 baseline — did
  either one actually get gates trending down toward the 0.2-0.3 range that
  would count as a real "surprise-only propagation is working" result?

**Did Model A's added complexity pay for itself?**
- Compare Model A's test accuracy AND parameter count against Model 2b —
  more machinery only matters if it beats a cheaper variant, not just the
  plain Baseline.

**Next steps if nothing here beats Baseline convincingly:** the honest
read, per the reckoning in `Brain-Inspired-Architecture-Research.md`, is
that this is a real empirical result worth writing up as-is ("we tried N
variants of surprise-gated propagation, here's what worked and didn't") —
not a reason to keep adding mechanisms until something sticks.